# Kapitel 19.2 - CGI Grundlagen und Formulare

In diesem Notebook lernst du CGI praktisch kennen.
Der Fokus liegt auf GET/POST, Formularfeldern, Validierung und robuster Ausgabe.

Wir simulieren mehrere Teile lokal in Python, damit du die Kernlogik verstehst, auch wenn kein echter Server aktiv ist.

# Lernziele

Nach diesem Notebook kannst du:

- den Unterschied zwischen GET und POST sauber erklaeren
- Query-Strings zerlegen und auswerten
- Formularwerte lesen und validieren
- HTML-Antworten strukturiert aufbauen
- typische Sicherheits- und Stabilitaetsprobleme frueh erkennen

# Voraussetzungen

Du solltest aus dem vorherigen Notebook bereits wissen:

- was ein HTTP-Request ist
- warum Header wichtig sind
- wie einfache dynamische Antworten entstehen

Zusatz: Grundlagen zu Dictionaries und Funktionen sind fuer die folgenden Aufgaben wichtig.

# Theorie

## GET

Bei GET stehen Daten in der URL, z. B. `?name=Ali&stadt=Berlin`.
Vorteile:
- leicht testbar
- bookmark-faehig

Nachteile:
- sichtbar in URL
- nicht fuer sensible Daten geeignet

## POST

Bei POST liegen Daten im Request-Body.
Vorteile:
- besser fuer Formulare mit groesseren Daten
- URL bleibt sauber

Wichtig: POST ist nicht automatisch sicher. Validierung und saubere Behandlung bleiben Pflicht.

# Erklaerung

In klassischem CGI bekommst du Daten ueber Umgebungsvariablen und Eingabestreams.
Typische Schluessel sind:

- `REQUEST_METHOD`
- `QUERY_STRING`
- `CONTENT_LENGTH`

Die Kernidee lautet:
1. Methode erkennen (GET/POST)
2. Datenquelle auswaehlen
3. Werte parsen
4. Werte validieren
5. HTML-Antwort erzeugen

# Syntax

## Minimales CGI-Skript

```python
print('Content-Type: text/html; charset=utf-8')
print()
print('<h1>Hallo aus CGI</h1>')
```

## Query-Parsing mit Standardbibliothek

```python
from urllib.parse import parse_qs
werte = parse_qs('name=Ali&stadt=Berlin')
```

# Merke

- In CGI ist die Ausgabe ueber `print` technisch Teil der HTTP-Antwort.
- `parse_qs` liefert Listen als Werte, weil Parameter mehrfach vorkommen koennen.
- Eingaben muessen immer validiert werden, auch wenn sie aus einem eigenen Formular stammen.

# Parameter

Wichtige Eingaben im CGI-Kontext:

- `QUERY_STRING`: GET-Parameter in Rohform
- `REQUEST_METHOD`: HTTP-Methode
- `CONTENT_LENGTH`: Laenge des POST-Bodys

Diese Daten sind oft Strings und muessen in passende Typen umgewandelt werden.

# Rueckgabewert

Ein CGI-Programm liefert am Ende Text auf `stdout`, der aus Header + Body besteht.
In Python modellieren wir das oft als String, den wir spaeter ausgeben.

Didaktischer Vorteil: Du kannst die Antwort zuerst im Notebook testen, bevor du sie im Server-Kontext nutzt.

In [ ]:
# Beispiel 1: GET-Parameter robust parsen
from urllib.parse import parse_qs

query_string = 'name=Elif&stadt=Koeln&hobby=lesen&hobby=schwimmen'
daten = parse_qs(query_string, keep_blank_values=True)

print(daten)
print('Name :', daten.get('name', [''])[0])
print('Stadt:', daten.get('stadt', [''])[0])
print('Hobbys:', daten.get('hobby', []))

# Beispiel 1 - Erklaerung

`parse_qs` gibt immer Listen zurueck.
Das ist korrekt, weil derselbe Parameter mehrfach vorkommen darf.
Bei Einzelfeldern liest du meist das erste Element (`[0]`).

In [ ]:
# Beispiel 2: Formularvalidierung (Name und Alter)
def validiere_formular(name, alter_text):
    fehler = []

    if not name.strip():
        fehler.append('Name darf nicht leer sein.')

    if not alter_text.isdigit():
        fehler.append('Alter muss eine ganze Zahl sein.')
    else:
        alter = int(alter_text)
        if alter < 0 or alter > 130:
            fehler.append('Alter muss zwischen 0 und 130 liegen.')

    return fehler

print(validiere_formular('Aylin', '27'))
print(validiere_formular('   ', '27'))
print(validiere_formular('Aylin', 'abc'))
print(validiere_formular('Aylin', '999'))

# Beispiel 2 - Erklaerung

Validierung ist keine Option, sondern Pflicht.
Sie schuetzt vor kaputten Daten und erleichtert spaeteres Debugging.
Je frueher fehlerhafte Eingaben erkannt werden, desto stabiler bleibt die Anwendung.

In [ ]:
# Beispiel 3: CGI-Antwortgenerator mit Erfolg/Fehler-Ansicht
def cgi_html_antwort(name, alter_text):
    fehler = validiere_formular(name, alter_text)

    if fehler:
        fehler_html = ''.join(f'<li>{f}</li>' for f in fehler)
        body = f'<h1>Fehler im Formular</h1><ul>{fehler_html}</ul>'
    else:
        body = f'<h1>Willkommen, {name}!</h1><p>Alter: {alter_text}</p>'

    header = 'Content-Type: text/html; charset=utf-8\n\n'
    html = f'<html><body>{body}</body></html>'
    return header + html

print(cgi_html_antwort('Mert', '31')[:150] + '...')
print(cgi_html_antwort('', 'abc')[:180] + '...')

# Beispiel 3 - Erklaerung

Dieses Muster ist in echten Anwendungen haeufig:
- zuerst validieren
- dann entweder Fehlermeldung oder Erfolgsausgabe erzeugen

Wichtig: Fehlermeldungen sollen klar sein, aber keine sensiblen Interna verraten.

# Praxisbeispiel

Wir bauen einen kleinen 'Anmelde-Controller' als reine Funktion.
Diese Funktion nimmt simulierte Request-Daten und liefert fertiges HTML zurueck.

In [ ]:
from urllib.parse import parse_qs

def verarbeite_anmeldung(query_string):
    daten = parse_qs(query_string, keep_blank_values=True)
    name = daten.get('name', [''])[0].strip()
    kurs = daten.get('kurs', ['Python'])[0].strip()

    if not name:
        body = '<h1>Anmeldung fehlgeschlagen</h1><p>Name fehlt.</p>'
    else:
        body = f'<h1>Anmeldung erfolgreich</h1><p>{name} nimmt am Kurs {kurs} teil.</p>'

    return 'Content-Type: text/html; charset=utf-8\n\n' + f'<html><body>{body}</body></html>'

print(verarbeite_anmeldung('name=Deniz&kurs=Webgrundlagen')[:160] + '...')
print(verarbeite_anmeldung('name=&kurs=Webgrundlagen')[:160] + '...')

# Haeufige Fehler

1. Query-String manuell mit `split('&')` ohne URL-Decoding.
2. Fehlende Feldnamen nicht mit Default-Werten absichern.
3. Unsichere String-Verarbeitung ohne Escaping bei HTML-Ausgabe.
4. Vermischung von Validierung, Business-Logik und HTML in einem riesigen Block.
5. POST-Daten wie GET behandeln (falsche Datenquelle).

# Best Practice

- Nutze `urllib.parse` statt selbstgebautem Parsing.
- Trenne Parsing, Validierung und Antworterzeugung in eigene Funktionen.
- Formuliere Fehlermeldungen benutzerfreundlich und praezise.
- Halte eine einheitliche Zeichenkodierung (`utf-8`) ein.
- Dokumentiere erwartete Felder klar (z. B. in Tabellen oder Docstrings).

# Tipp

Wenn du komplexere Formulare baust, erstelle frueh ein strukturiertes Datenobjekt,
z. B. ein Dictionary mit bereits bereinigten Werten.
So wird dein spaeterer Code deutlich wartbarer.

# Uebung

Baue eine Funktion `bewerte_feedback(query_string)` mit folgenden Regeln:

- Felder: `name`, `bewertung`, `kommentar`
- `bewertung` muss Zahl 1 bis 5 sein
- Name darf nicht leer sein
- Gib bei Fehlern eine HTML-Liste mit Meldungen zurueck
- Bei Erfolg gib eine Danke-Seite zurueck

In [ ]:
# Loesung
from urllib.parse import parse_qs

def bewerte_feedback(query_string):
    daten = parse_qs(query_string, keep_blank_values=True)
    name = daten.get('name', [''])[0].strip()
    bewertung_text = daten.get('bewertung', [''])[0].strip()
    kommentar = daten.get('kommentar', [''])[0].strip()

    fehler = []
    if not name:
        fehler.append('Name fehlt.')

    if not bewertung_text.isdigit():
        fehler.append('Bewertung muss numerisch sein.')
    else:
        bewertung = int(bewertung_text)
        if not (1 <= bewertung <= 5):
            fehler.append('Bewertung muss zwischen 1 und 5 liegen.')

    if fehler:
        body = '<h1>Feedback nicht gespeichert</h1><ul>' + ''.join(f'<li>{f}</li>' for f in fehler) + '</ul>'
    else:
        body = f'<h1>Danke, {name}!</h1><p>Bewertung: {bewertung_text}/5</p><p>Kommentar: {kommentar}</p>'

    return 'Content-Type: text/html; charset=utf-8\n\n' + f'<html><body>{body}</body></html>'

print(bewerte_feedback('name=Ece&bewertung=5&kommentar=Sehr%20gut')[:170] + '...')
print(bewerte_feedback('name=&bewertung=9&kommentar=test')[:170] + '...')

# Zusammenfassung

Du hast in diesem Notebook die CGI-Praxis intensiv geuebt:

- GET/POST konzeptionell verstanden
- Query-Strings robust geparst
- Formulareingaben validiert
- HTML-Antworten dynamisch erzeugt

Im naechsten Teil vertiefen wir CGI mit Fokus auf Sicherheit, Struktur und kleinen Realprojekten.

# Weiterfuehrende Links

- Python `urllib.parse` Dokumentation
- Grundlagen zu URL-Encoding
- OWASP Input Validation Cheat Sheet

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.